In [22]:
import pandas as pd

import great_expectations as gx
import synapseclient

from agoradatatools.gx import GreatExpectationsRunner

context = gx.get_context(project_root_dir='../src/agoradatatools/great_expectations')

from expectations.expect_column_nested_field_values_mostly_meet_string_requirement import ExpectColumnMostlyStringLength
    

# Create Expectation Suite for UI Config Data

## Get Example Data File

In [23]:
syn = synapseclient.Synapse()
syn.login()


UPGRADE AVAILABLE

A more recent version of the Synapse Client (4.9.0) is available. Your version (4.4.1) can be upgraded by typing:
    pip install --upgrade synapseclient

Python Synapse Client version 4.9.0 release notes

https://python-docs.synapse.org/news/



Welcome, Lingling Peng!

INFO: 2025-08-04 15:01:08 | synapseclient_default | Welcome, Lingling Peng!



In [24]:
ui_config_data_file = syn.get("syn66531901").path

## Create Validator Object on Data File

In [25]:
df = pd.read_json(ui_config_data_file)
nested_columns = ["columns"]
df = GreatExpectationsRunner.convert_nested_columns_to_json(df, nested_columns)
validator = context.sources.pandas_default.read_dataframe(df)
validator.expectation_suite_name = "ui_config"

## Add Expectations to Validator Object For Each Column

In [26]:
validator.expect_column_mostly_string_length(column="columns", target_field="tooltip", operator=">", length_threshold=0, valid_string_threshold=0.35)

[WARNING] /Users/lpeng/.local/share/virtualenvs/agora-data-tools-asv3WV1c/lib/python3.10/site-packages/great_expectations/expectations/expectation.py:1481: UserWarning: `result_format` configured at the Validator-level will not be persisted. Please add the configuration to your Checkpoint config or checkpoint_run() method instead.
  warnings.warn(



  warnings.warn(



Calculating Metrics:   0%|          | 0/4 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "observed_valid_ratio": 0.36
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

## Save Expectation Suite

In [27]:
validator.save_expectation_suite(discard_failed_expectations=False)

## Create Checkpoint and View Results

In [28]:
checkpoint = context.add_or_update_checkpoint(
    name="agora-test-checkpoint",
    validator=validator,
)
checkpoint_result = checkpoint.run()
context.view_validation_result(checkpoint_result)

Calculating Metrics:   0%|          | 0/4 [00:00<?, ?it/s]

## Build Data Docs - Click on Expectation Suite to View All Expectations

In [29]:
context.build_data_docs()
context.open_data_docs()